# Checking my own subject list against what else CDA knows

*Standalone -- no prior vignette required.*

> **DRAFT STATUS:** this notebook has not yet been run against the live API. The file (`mydatafile.tsv`) and narrative framing are confirmed real; specific match counts and source breakdowns below are still placeholders.

I've got a cohort I've assembled over time from a few different places -- some TCGA subjects, some CPTAC, some HTAN -- and I've already been doing my own analysis on them. My working file has a z-score column I calculated myself sitting right next to the subject IDs. I'm not looking for new subjects, and I'm not trying to explore anything -- I already have my list. What I want to know is whether there's more data about these specific people sitting somewhere in CDA, without retyping every ID into a filter by hand, and *without making a separate clean file just for CDA* -- I want to use the file I already have, extra columns and all.

In [ ]:
from cdapython import *
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt
opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0
opt.columnDefs = [{"className": "dt-left", "targets": "_all"}]

My file, `mydatafile.tsv`, looks like this:

```
subject	zscore
TCGA.TCGA-C8-A12K	0.41
TCGA.TCGA-BH-A0B8	1.01
htan_wustl.HTA12_1	3.56
CPTAC.06BR003	1.47
...
```

`match_from_file` takes three things: which file, which column in that file to use, and which CDA column to match it against. It doesn't care that there's a second column in there I'm not matching on -- `zscore` isn't part of the lookup at all, it's just along for the ride in my own file:

In [ ]:
get_subject_data(
    match_from_file={
        'input_file': 'mydatafile.tsv',
        'input_column': 'subject',
        'cda_column_to_match': 'subject_id'
    },
    add_columns='upstream_identifiers.*'
)

This works whether the IDs in my file are CDA's own subject IDs or each source's original native IDs -- as long as the subject actually came from one of the data centers CDA aggregates, this matches either way. I didn't need to look up or convert anything first.

Adding `upstream_identifiers.*` shows me exactly which sources have data on each subject. Before pulling full records, I want a shape of what I'd actually be getting:

In [ ]:
summarize_files(match_from_file={
    'input_file': 'mydatafile.tsv',
    'input_column': 'subject',
    'cda_column_to_match': 'subject_id'
})

In [ ]:
summarize_subjects(match_from_file={
    'input_file': 'mydatafile.tsv',
    'input_column': 'subject',
    'cda_column_to_match': 'subject_id'
})

Now that I know there's more data out there on these people than what I'd already assembled myself, I want to know exactly where each piece came from -- not just "GC has something," but a pointer back to that source's own record for this specific subject, so I can look at it directly or cite it properly:

In [ ]:
subjects_with_refs = get_subject_data(
    match_from_file={
        'input_file': 'mydatafile.tsv',
        'input_column': 'subject',
        'cda_column_to_match': 'subject_id'
    },
    include_external_refs=True
)
subjects_with_refs

There's a new `external_reference_data` column here -- a small table per subject pointing back to the actual upstream record that described them, separate from `data_source` (which just tells me *which* data center a row came from, not the specific record). This is the kind of thing I'd want before citing a source properly in a publication, or before trusting that CDA's version of a record matches what the original data center actually says.

And my `zscore` column never had to leave my spreadsheet in the first place -- I searched using the file I already had, not a stripped-down copy made just for this lookup.